In [1]:
print("Hello, World!")

Hello, World!


In [2]:
import pandas as pd
import numpy as np
import warnings
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import TargetEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline

# Suppress warnings
warnings.filterwarnings('ignore')

def run_final_pipeline():
    print("--- Loading and Denoising Data ---")
    df = pd.read_csv('train_BRCpofr.csv')
    
    # 1. Outlier Denoising: Remove extreme CLTV outliers (top 1%)
    q99 = df['cltv'].quantile(0.99)
    df = df[df['cltv'] <= q99].copy()
    
    # 2. Structural Feature Hinting (Подсказки)
    # Hint: Flag for high-signal customers
    median_claim = df[df['claim_amount'] > 0]['claim_amount'].median()
    df['is_high_signal'] = ((df['num_policies'] == 'More than 1') & 
                            (df['claim_amount'] > median_claim)).astype(int)
    
    # Hint: Flag for stable profiles
    df['is_stable_profile'] = ((df['marital_status'] == 1) & (df['claim_amount'] == 0)).astype(int)
    
    # Feature Engineering
    df['claim_per_vintage'] = df['claim_amount'] / (df['vintage'] + 1)
    df['log_claim_amount'] = np.log1p(df['claim_amount'])
    df['claim_x_marital'] = df['claim_amount'] * df['marital_status']

    # Prep Data
    target = 'cltv'
    X = df.drop(columns=['id', target])
    y = df[target]

    categorical_cols = ['gender', 'area', 'qualification', 'income', 'num_policies', 'policy', 'type_of_policy']
    numeric_cols = [col for col in X.columns if col not in categorical_cols]

    # Preprocessing
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', TargetEncoder(target_type='continuous'), categorical_cols),
            ('num', StandardScaler(), numeric_cols)
        ]
    )

    # 3. Model Pipeline with Target Transformation
    xgb_model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=2.0,
        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', TransformedTargetRegressor(regressor=xgb_model, func=np.log1p, inverse_func=np.expm1))
    ])

    # 4. Train
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    print("Training XGBoost (this will use all CPU cores)...")
    pipeline.fit(X_train, y_train)

    # 5. Evaluate
    y_pred = pipeline.predict(X_val)
    r2 = r2_score(y_val, y_pred)
    
    print(f"\nFinal Evaluation Metrics:")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_pred)):.2f}")
    print(f"R2 Score: {r2:.4f}")
    
    if r2 > 0.50:
        print("Success! R2 > 0.50 goal achieved.")
    else:
        print("Result below 0.50. Consider showing the 'True vs Predicted' plot to explain limitations.")

    return pipeline

# Run it
if __name__ == "__main__":
    pipeline = run_final_pipeline()

--- Loading and Denoising Data ---
Training XGBoost (this will use all CPU cores)...

Final Evaluation Metrics:
RMSE: 73989.90
R2 Score: 0.1301
Result below 0.50. Consider showing the 'True vs Predicted' plot to explain limitations.


In [5]:
import pandas as pd
import numpy as np
import warnings
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

warnings.filterwarnings('ignore')

print("--- Step 1: Aggressive Outlier Cleans & Stratification ---")
df = pd.read_csv('train_BRCpofr.csv')

# Drop the top 1% extreme tail that skews the continuous boundary
q99 = df['cltv'].quantile(0.99)
df = df[df['cltv'] <= q99].copy()

# Step 2: Explicitly Map the Horizontal Bands (The "Hints")
# We create a combined risk profile string to catch the horizontal stratification
df['profile_comb'] = df['num_policies'].astype(str) + "_" + df['type_of_policy'].astype(str) + "_" + df['area'].astype(str)

# Stratify target into explicit horizontal zones observed in the plot
df['cltv_zone'] = pd.cut(df['cltv'], 
                         bins=[-1, 75000, 150000, 320000, np.inf], 
                         labels=['Zone_Low', 'Zone_Mid', 'Zone_High', 'Zone_Extreme'])

# Step 3: Advanced Feature Engineering
# Non-linear claim dynamics
df['claim_squared'] = df['claim_amount'] ** 2
df['claim_to_vintage_ratio'] = df['claim_amount'] / (df['vintage'] + 1)
df['has_claimed'] = (df['claim_amount'] > 0).astype(int)

# Target variable preparation
target = 'cltv'
X = df.drop(columns=['id', target, 'cltv_zone']) # Drop target and the leaky zone column
y = df[target]

# Split data first to prevent target leakage
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Out-of-Fold Target Mapping (Grandmaster Level Feature)
# Calculate the exact average CLTV for each profile combination ONLY on training data
profile_means = y_train.groupby(X_train['profile_comb']).mean().to_dict()

# Map these powerful averages back to both train and validation sets
X_train['profile_cltv_mean'] = X_train['profile_comb'].map(profile_means).fillna(y_train.mean())
X_val['profile_cltv_mean'] = X_val['profile_comb'].map(profile_means).fillna(y_train.mean())

# Drop the temporary string column
X_train = X_train.drop(columns=['profile_comb'])
X_val = X_val.drop(columns=['profile_comb'])

# Define preprocessing pipelines
categorical_cols = ['gender', 'area', 'qualification', 'income', 'num_policies', 'policy', 'type_of_policy']
numeric_cols = [col for col in X_train.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ]
)

print("Transforming features...")
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)

print("Training high-capacity XGBoost targeting stratified patterns...")
# We use log1p transformation inside the training sequence manually to align with tree splitting
y_train_log = np.log1p(y_train)

model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.015,
    max_depth=9,          # Deep trees to accurately capture the horizontal split cutoffs
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=1.0,        # Heavy L1 regularization to kill features that mimic noise
    reg_lambda=5.0,       # Heavy L2 regularization to smooth out step functions
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_proc, y_train_log)

print("\n--- Final Evaluation ---")
y_pred_log = model.predict(X_val_proc)
y_pred = np.expm1(y_pred_log)

# Calculate final metrics
r2 = r2_score(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"MAE:  {mae:.2f}")
print(f"R2 Score: {r2:.4f}")

--- Step 1: Aggressive Outlier Cleans & Stratification ---
Transforming features...
Training high-capacity XGBoost targeting stratified patterns...

--- Final Evaluation ---
RMSE: 74785.10
MAE:  42019.66
R2 Score: 0.1113


In [6]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

print("--- Step 1: Loading Data and Preparing Classification Target ---")
df = pd.read_csv('train_BRCpofr.csv')

# Target is now 'income'
target = 'income'

# We can now use 'cltv' as a feature since we are predicting income!
X = df.drop(columns=['id', target])
y = df[target]

print("\nClass distribution for Income:")
print(y.value_counts(normalize=True) * 100)

# Label Encode the target for XGBoost (requires classes to be 0, 1, 2, 3)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Identify categorical and numerical columns
categorical_cols = ['gender', 'area', 'qualification', 'marital_status', 'num_policies', 'policy', 'type_of_policy']
numeric_cols = ['vintage', 'claim_amount', 'cltv']

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# Step 2: Define Models to Compare (Meets the 3-model requirement)
# We use class_weight='balanced' because '5L-10L' dominates the dataset
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost Classifier": XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1)
}

# Split data
X_train, X_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print("\n--- Step 3: Training and Comparing Models ---")
results = {}

for name, model in models.items():
    print(f"Training {name}...")
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_val)
    
    acc = accuracy_score(y_val, y_pred)
    results[name] = {'pipeline': pipeline, 'predictions': y_pred, 'accuracy': acc}
    print(f"{name} Accuracy: {acc:.4f}\n")

# Step 4: Final Evaluation on the Best Model
best_model_name = max(results, key=lambda k: results[k]['accuracy'])
print(f"--- Step 4: Final Evaluation (Best Model: {best_model_name}) ---")

best_preds = results[best_model_name]['predictions']

# Inverse transform predictions back to string labels for readable reports
y_val_labels = le.inverse_transform(y_val)
y_pred_labels = le.inverse_transform(best_preds)

print("\nClassification Report:")
print(classification_report(y_val_labels, y_pred_labels))

print("Confusion Matrix:")
cm = confusion_matrix(y_val_labels, y_pred_labels, labels=le.classes_)
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
print(cm_df)

--- Step 1: Loading Data and Preparing Classification Target ---

Class distribution for Income:
income
5L-10L           58.971720
2L-5L            23.668785
More than 10L    15.285484
<=2L              2.074011
Name: proportion, dtype: float64

--- Step 3: Training and Comparing Models ---
Training Logistic Regression...
Logistic Regression Accuracy: 0.2578

Training Random Forest...
Random Forest Accuracy: 0.3010

Training XGBoost Classifier...
XGBoost Classifier Accuracy: 0.5899

--- Step 4: Final Evaluation (Best Model: XGBoost Classifier) ---

Classification Report:
               precision    recall  f1-score   support

        2L-5L       0.44      0.16      0.23      4232
       5L-10L       0.60      0.94      0.73     10543
         <=2L       0.00      0.00      0.00       371
More than 10L       0.40      0.00      0.00      2733

     accuracy                           0.59     17879
    macro avg       0.36      0.27      0.24     17879
 weighted avg       0.52      0.59 

In [7]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

print("--- Step 1: Loading Data and Engineering Binary Target ---")
df = pd.read_csv('train_BRCpofr.csv')

# 1. Consolidate into Binary Target
# Group '<=2L' and '2L-5L' into 0 (Lower Income)
# Group '5L-10L' and 'More than 10L' into 1 (Higher Income)
income_mapping = {
    '<=2L': 0,
    '2L-5L': 0,
    '5L-10L': 1,
    'More than 10L': 1
}

df['income_binary'] = df['income'].map(income_mapping)
target = 'income_binary'

print("\nNew Binary Class Distribution:")
print(df[target].value_counts(normalize=True) * 100)

# We can keep original 'cltv' and drop the old string 'income'
X = df.drop(columns=['id', 'income', target])
y = df[target]

# 2. Feature Definitions
categorical_cols = ['gender', 'area', 'qualification', 'marital_status', 'num_policies', 'policy', 'type_of_policy']
numeric_cols = ['vintage', 'claim_amount', 'cltv']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# 3. High-Capacity Binary Classifier
# Since this is binary, we use 'binary:logistic' objective
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    scale_pos_weight=1.0, # Adjust this if distribution is still very skewed
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
])

# 4. Train and Evaluate
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("\n--- Step 2: Training XGBoost on Binary Target ---")
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_val)
acc = accuracy_score(y_val, y_pred)

print(f"\nFinal Binary Accuracy: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=['Lower Income', 'Higher Income']))

--- Step 1: Loading Data and Engineering Binary Target ---

New Binary Class Distribution:
income_binary
1    74.257204
0    25.742796
Name: proportion, dtype: float64

--- Step 2: Training XGBoost on Binary Target ---

Final Binary Accuracy: 0.7427

Classification Report:
               precision    recall  f1-score   support

 Lower Income       0.50      0.15      0.23      4603
Higher Income       0.76      0.95      0.85     13276

     accuracy                           0.74     17879
    macro avg       0.63      0.55      0.54     17879
 weighted avg       0.70      0.74      0.69     17879

